# Titanic disaster

Explore passengers, think about which features are informative, compare logistic regression with a random forest, and check your choice on unseen passengers. Follow the [problem](README.md).

The data are all 1,309 labelled historical titanic3 passengers, prepared for you by `load_passengers()` in [titanic.py](titanic.py): it returns a pandas DataFrame with numeric columns converted and blanks marked as missing. See the [source note](data/TITANIC_SOURCE.md).

Run the setup cell first. Leave `X_test` and `y_test` alone until the final section: everything before that uses `X_train` and `y_train`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

from exercises.titanic_disaster.titanic import (
    load_passengers,
    titanic_pipeline,
)

passengers = load_passengers()
features = passengers.drop(columns="Survived")
target = passengers["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=42
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 1. Meet the passengers

Explore the training set. Before running the example, write down which features you expect to help.

Useful patterns:

- `isna().sum()` counts missing values.
- `groupby(...)["Survived"].agg(["count", "mean"])` summarises survival within each group. The mean of a 0/1 outcome is the proportion who survived.

One example is shown below. Add plots and summaries for age, fare and family counts. Which features might repeat information?

In [ ]:
# TODO: extend these training-only examples.
training = X_train.join(y_train)
print(training.isna().sum())
print(training.groupby("Pclass")["Survived"].agg(["count", "mean"]))
training.groupby("Sex")["Survived"].mean().plot.bar(ylabel="Survival proportion")
plt.show()

## 2. Build the classifiers

In [titanic.py](titanic.py), `make_preprocessor()` supplies the preprocessing. Write `titanic_pipeline` to chain it with logistic regression or a random forest, as described in the problem, then check with `uv run pytest exercises/data_analysis/titanic/ -q`.

## 3. Compare using cross-validation

`cross_val_score` splits the training rows into the five folds in `cv`. For each fold it fits a fresh copy of the model on the other four and measures **accuracy**, the fraction of passengers predicted correctly, on the held-out fold. It returns one score per fold.

Because we pass the whole pipeline, imputation and encoding are also learned from each fold's training rows only.

The baseline always predicts the most common outcome: it is the score to beat. Add your two pipelines, using each classifier's default settings, and compare their mean accuracy. The only argument to pass is `random_state=42`, which makes the random forest's results repeatable.

In [ ]:
models = {
    "baseline": DummyClassifier(strategy="most_frequent"),
    # TODO: add "logistic" pipeline
    # TODO: add "forest" pipeline
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv)
    print(f"{name}: mean accuracy {scores.mean():.3f} (std {scores.std():.3f})")

Did the logistic regression model or random forest model beat the baseline? Which would you choose, and why?

### Optional extension 🚀: does tuning help?

The comparison above used each classifier's default settings. Because `titanic_pipeline` passes extra keyword arguments to the classifier, you can reuse it to try other settings. For the random forest, `n_estimators` is the number of trees. Choose some values to try below and compare their cross-validation accuracy using the same folds.

Does accuracy keep improving as you add trees? Is the improvement substantial enough to change your choice, or are the results broadly similar? You could also try another setting, such as `max_depth`.

Choose using validation accuracy only: **do not use the test set**. If you choose tuned settings, add them to `extra_args` in section 4.

In [ ]:
candidates = []  # TODO: choose some values of n_estimators to try

for n_estimators in candidates:
    model = titanic_pipeline("forest", n_estimators=n_estimators, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv)
    print(
        f"n_estimators={n_estimators}: mean accuracy {scores.mean():.3f} (std {scores.std():.3f})"
    )

## 4. Evaluate separately

Only now use the test set, and only once. 

- `fit(X_train, y_train)` learns from all training rows.
- `score(X_test, y_test)` predicts the test passengers and returns the accuracy against their true outcomes. 

**Do not go back and retune after seeing this result.**

In [ ]:
# TODO: set chosen_model to "logistic" or "forest", based on your validation results.
chosen_model = None
extra_args = {"random_state": 42}  # Add any tuned classifier settings here.

final_model = titanic_pipeline(chosen_model, **extra_args).fit(X_train, y_train)
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"baseline test accuracy: {baseline.score(X_test, y_test):.3f}")
print(f"{chosen_model} test accuracy: {final_model.score(X_test, y_test):.3f}")

Write a short conclusion: 
- which features seemed informative? 
- which model you chose? 
- did test accuracy match what cross-validation suggested? 